In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
import shap
from scipy.sparse import csr_matrix, hstack

In [ ]:
best_model = joblib.load("best_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")
explainer = shap.TreeExplainer(best_model)

In [ ]:
HANDCRAFTED = [
"char_count",
"word_count",
"sentence_count",
"avg_word_length",
"digit_ratio",
"special_ratio",
"exclamation_count",
"question_count",
"has_email",
"gmail_email",
"yahoo_email",
"outlook_email",
"salary_present",
"salary_daily",
"salary_monthly",
"salary_hourly",
"avg_sentence_length",
"unique_word_ratio",
"repetition_score",
"long_word_ratio",
"numeric_token_ratio"
]

In [ ]:
def extract_features(text):

    words = text.split()

    char_count = len(text)

    word_count = len(words)

    sentence_count = max(
        len(re.findall(r"[.!?]", text)),
        1
    )

    avg_word_length = np.mean(
        [len(i) for i in words]
    ) if words else 0

    digit_ratio = sum(
        c.isdigit()
        for c in text
    ) / max(len(text),1)

    special_ratio = len(
        re.findall(
            r"[^a-zA-Z0-9\s]",
            text
        )
    ) / max(len(text),1)

    exclamation_count = text.count("!")

    question_count = text.count("?")

    has_email = int(
        bool(
            re.search(
                r"\S+@\S+",
                text
            )
        )
    )

    gmail_email = int(
        "@gmail" in text.lower()
    )

    yahoo_email = int(
        "@yahoo" in text.lower()
    )

    outlook_email = int(
        "@outlook" in text.lower()
    )

    salary_present = int(
        bool(
            re.search(
                r"\d",
                text
            )
        )
    )

    salary_daily = int(
        "day" in text.lower()
    )

    salary_monthly = int(
        "month" in text.lower()
    )

    salary_hourly = int(
        "hour" in text.lower()
    )

    avg_sentence_length = min(
        word_count/sentence_count,
        100
    )

    unique_word_ratio = len(
        set(words)
    ) / max(word_count,1)

    repetition_score = (
        1-unique_word_ratio
    )

    long_word_ratio = sum(
        len(i)>8
        for i in words
    ) / max(word_count,1)

    numeric_token_ratio = sum(
        any(c.isdigit() for c in i)
        for i in words
    ) / max(word_count,1)

    return [

        char_count,

        word_count,

        sentence_count,

        avg_word_length,

        digit_ratio,

        special_ratio,

        exclamation_count,

        question_count,

        has_email,

        gmail_email,

        yahoo_email,

        outlook_email,

        salary_present,

        salary_daily,

        salary_monthly,

        salary_hourly,

        avg_sentence_length,

        unique_word_ratio,

        repetition_score,

        long_word_ratio,

        numeric_token_ratio

    ]

In [ ]:
def explain_job(

    job_title,

    company_name,

    job_description

):

    combined = (

        str(job_title)

        + " "

        + str(company_name)

        + " "

        + str(job_description)

    )

    tfidf_vector = tfidf.transform(

        [combined]

    )

    manual = csr_matrix(

        [extract_features(combined)]

    )

    X = hstack([

        tfidf_vector,

        manual

    ])

    prob = float(

        best_model.predict_proba(X)[0][1]

    )

    pred = int(

        best_model.predict(X)[0]

    )

    fraud=[]

    safe=[]

    lower=combined.lower()

    if "whatsapp" in lower:
        fraud.append(
            "WhatsApp contact detected"
        )

    if "telegram" in lower:
        fraud.append(
            "Telegram contact detected"
        )

    if "registration fee" in lower:
        fraud.append(
            "Registration fee requested"
        )

    if "joining fee" in lower:
        fraud.append(
            "Joining fee mentioned"
        )

    if "security deposit" in lower:
        fraud.append(
            "Security deposit requested"
        )

    if "@gmail" in lower:
        fraud.append(
            "Personal Gmail recruiter email"
        )

    if "earn daily" in lower:
        fraud.append(
            "Unrealistic earning pattern"
        )

    if "captcha" in lower:
        fraud.append(
            "Captcha typing related content"
        )

    if "crypto" in lower:
        fraud.append(
            "Cryptocurrency payment mentioned"
        )

    if len(fraud)==0:

        safe.append(
            "No obvious scam phrases detected"
        )

        safe.append(
            "Professional job structure detected"
        )

    risk="LOW"

    if prob>0.90:

        risk="HIGH"

    elif prob>0.60:

        risk="MEDIUM"

    trust=round((1-prob)*100)

    return {

        "Prediction":

        "FAKE"

        if pred==1

        else

        "REAL",

        "Confidence":

        round(

            max(

                prob,

                1-prob

            )*100,

            2

        ),

        "Risk":

        risk,

        "Trust Score":

        trust,

        "Fraud Indicators":

        fraud,

        "Safe Indicators":

        safe

    }

In [ ]:
result = explain_job(

job_title="Python Developer",

company_name="Google",

job_description="""

We are hiring a Python Developer.

Remote work available.

Salary ₹12 LPA.

Apply through official careers portal.

"""

)

result

In [ ]:
print(type(result))
print(result)

In [ ]:
result = explain_job(

job_title="Work From Home Data Entry",

company_name="ABC Digital",

job_description="""

Earn ₹5000 Daily.

No Interview.

WhatsApp HR now.

Registration fee ₹499.

Limited Seats.

"""

)

print(result)

In [ ]:
result = explain_job(

job_title="Software Engineer",

company_name="Microsoft",

job_description="""

We are looking for an experienced Software Engineer.

Responsibilities include backend development,
system design and cloud deployment.

Apply only through our official careers portal.

"""

)

print(result)

In [ ]:
tfidf_features = tfidf.get_feature_names_out()

manual_features = [

    "char_count",
    "word_count",
    "sentence_count",
    "avg_word_length",
    "digit_ratio",
    "special_ratio",
    "exclamation_count",
    "question_count",

    "has_email",
    "gmail_email",
    "yahoo_email",
    "outlook_email",

    "salary_present",
    "salary_daily",
    "salary_monthly",
    "salary_hourly",

    "avg_sentence_length",
    "unique_word_ratio",
    "repetition_score",
    "long_word_ratio",
    "numeric_token_ratio"

]

feature_names = list(tfidf_features) + manual_features

In [ ]:
feature_mapping = {

"gmail_email":"Personal Gmail recruiter email detected",
"yahoo_email":"Yahoo recruiter email detected",
"outlook_email":"Outlook recruiter email detected",
"salary_daily":"Daily payment pattern detected",
"salary_hourly":"Hourly payment pattern detected",
"salary_present":"Salary information present",
"digit_ratio":"High amount of numeric information",
"special_ratio":"Spam-like special characters detected",
"question_count":"Persuasive questioning language",
"exclamation_count":"Aggressive promotional language",
"repetition_score":"Highly repetitive writing",
"unique_word_ratio":"Rich vocabulary",
"char_count":"Detailed job description",
"word_count":"Long job description",
"sentence_count":"Structured job description"
}

In [ ]:
def explain_sample(index):
    sample = X_test[index]
    prob = best_model.predict_proba(sample)[0][1]
    pred = best_model.predict(sample)[0]
    shap_values = explainer.shap_values(sample)

    shap_df = pd.DataFrame({
        "feature":feature_names,
        "value":shap_values[0]
    })

    shap_df["abs"] = shap_df["value"].abs()
    shap_df = shap_df.sort_values(
        by="abs",
        ascending=False
    )

    positive=[]
    negative=[]

    for _,row in shap_df.head(15).iterrows():
        feat=row["feature"]
        text=feature_mapping.get(
            feat,
            feat.replace("_"," ")
        )

        if row["value"]>0:
            positive.append(text)
        else:
            negative.append(text)

    risk="LOW"

    if prob>0.90:
        risk="HIGH"
    elif prob>0.60:
        risk="MEDIUM"

    return {
        "Prediction":
        "FAKE"

        if pred==1
        else
        "REAL",
        "Confidence":

        round(
            max(prob,1-prob)*100,2),

        "Risk":
        risk,
        "Positive Signals":
        positive,
        "Negative Signals":
        negative
    }

In [ ]:
import joblib

X_test = joblib.load("X_test.pkl")
y_test = joblib.load("y_test.pkl")

In [ ]:
result = explain_sample(0)
result